# 05 — Re-train Defense-Prefix (3 seeds + 2 controls) and evaluate every vector

Re-implements the authors' training on the cached ImageNet-100 features (same initialisation, 81 templates,
CE + 3·KL, SGD 0.002, cosine to 5e-5, batch 512, 10 epochs), with seeds 0, 1, 2, plus two controls:
identity loss only (the paper's "w/o defense loss") and an untrained random vector.
Then evaluates the authors' vector and all of ours on every cached test set and on the seen/unseen sets of notebook 04.

**Before running:** Runtime → **T4 GPU**. Drive must contain `MyDrive/dp_project/cache_train/train_imagenet100.pt`
and `MyDrive/dp_project/cache_test/*.pt`. Add `cars.pt` and `imagenet.pt` (notebook 02-2) to `cache_test/` when ready
and re-run part D only. Finished seeds are skipped, so the notebook can be re-run after a disconnect.

In [1]:
# A1) Paths and environment (Colab: everything is read from and written to Google Drive)
from google.colab import drive
drive.mount("/content/drive")
import glob, os, sys, subprocess, json
DRIVE = "/content/drive/MyDrive/dp_project"
W = f"{DRIVE}/experiments"; BASE = "/content"          # results go to Drive and survive the session
os.makedirs(W, exist_ok=True)
TRAIN = f"{DRIVE}/cache_train/train_imagenet100.pt"
CACHE = f"{DRIVE}/cache_test"                     # cars.pt / imagenet.pt from Step 2c: copy them into this folder
assert os.path.exists(TRAIN), "upload train_imagenet100.pt to MyDrive/dp_project/cache_train/"
print("train file:", TRAIN); print("test sets found:", len(glob.glob(f"{CACHE}/*.pt")), sorted(os.listdir(CACHE)))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
train file: /content/drive/MyDrive/dp_project/cache_train/train_imagenet100.pt
test sets found: 10 ['aircraft.pt', 'caltech.pt', 'disentangling.pt', 'dtd.pt', 'eurosat.pt', 'flowers.pt', 'food.pt', 'paint.pt', 'pets.pt', 'rta-100.pt']
Tesla T4, 15360 MiB


In [2]:
# A2) Dependencies, authors' code pinned to 50d7887, the two compatibility patches (same as Steps 1-2)
!pip -q install ftfy regex git+https://github.com/openai/CLIP.git
import pathlib
os.chdir(BASE)
if not os.path.isdir("Defense-Prefix"):
    subprocess.run("git clone -q https://github.com/azuma164/Defense-Prefix.git", shell=True, check=True)
os.chdir(f"{BASE}/Defense-Prefix")
subprocess.run("git checkout -q 50d7887", shell=True, check=True)
p = pathlib.Path("datasets/utils/make_dataset_train.py"); s = p.read_text()
if "_textsize" not in s:
    helper = "\ndef _textsize(draw, text, font):\n    # Replacement for ImageDraw.textsize (removed in Pillow 10).\n    l, t, r, b = draw.multiline_textbbox((0, 0), text, font=font)\n    return r, b\n"
    s = s.replace("random.seed(300)", "random.seed(300)\n" + helper, 1).replace("imagedraw.textsize(text, font=font)", "_textsize(imagedraw, text, font)")
    p.write_text(s)
e = pathlib.Path("eval.py"); s = e.read_text()
e.write_text(s.replace("torch.load(args.tokens_path)", "torch.load(args.tokens_path, map_location=device, weights_only=False)"))
import torch, torchvision, PIL
print(subprocess.run("git rev-parse HEAD && git diff --stat", shell=True, capture_output=True, text=True).stdout)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__, "| Pillow", PIL.__version__)
!pip freeze > {W}/pip_freeze_step3_4.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
50d7887017b295e76b15e56c137ab47e68264448
 datasets/utils/make_dataset_train.py | 10 ++++++++--
 eval.py                              |  2 +-
 2 files changed, 9 insertions(+), 3 deletions(-)

python 3.13.16 | torch 2.14.1+cu130 | torchvision 0.29.1+cu130 | Pillow 11.3.0


### A3) Our scripts (written into the authors' repo root so they can import `utils/`, `templates.py`, `datasets/`)

In [3]:
%%writefile train_dp_cached.py
"""Train a Defense-Prefix (DP) vector on cached CLIP image features.

Re-implements the authors' train.py (commit 50d7887) with one change: image features
are read from `train_imagenet100.pt` instead of being recomputed every step.  In the
authors' code the image features are computed under `torch.no_grad()` from a frozen
encoder, so caching them gives the same loss and the same gradients for the DP vector.

Kept identical to the authors' train.py:
  * CLIP ViT-B/32, loaded with clip.load (fp16 on GPU), frozen
  * DP vector: 1 x 512, initialised N(0, 0.02), inserted in place of the "*" token
    with the authors' encode_text_with_learnt_tokens (utils/non_nv.py)
  * one random template (of the 81 in templates.py) per batch, shared by all classes
  * loss = seta * CE(attacked images, DP class features) + gamma * KL(p_orig || p_dp) on clean images
    seta = 1, gamma = 3, logits scaled by model.logit_scale.exp()
  * SGD lr 0.002, CosineAnnealingLR(T_max=epochs, eta_min=5e-5) stepped once per epoch
  * batch size 512, 10 epochs, shuffled, last partial batch kept

Small numerical difference (recorded as a deviation): the CE and KL terms are computed in
fp32 from the fp16 logits (the authors compute them directly on fp16 logits).

Added (all optional, defaults reproduce the paper):
  --seed                 seeds python/numpy/torch and the shuffle order (the authors set no seed)
  --dp_dtype fp16|fp32   authors' DP parameter is fp16 (model.dtype); fp32 is offered as a deviation
  --no_defense / --no_reg  ablations: identity loss only / defense loss only
  --extra_words FILE     extra attack words added as distractor classes (for the broad-vocabulary run)
  If `typographic` in the training file has shape (K, N, 512), one of the K attacked
  versions is drawn per epoch (attack renewal).

Usage (from the Defense-Prefix repo root, so `utils/` and `templates.py` import):
  python train_dp_cached.py --train_file /path/train_imagenet100.pt --out runs/dp_seed0 --seed 0
"""
import argparse
import json
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
import clip  # noqa: E402
from templates import imagenet_templates  # noqa: E402  (authors' 81 templates)
from utils.non_nv import encode_text_with_learnt_tokens  # noqa: E402  (authors' function)

device = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_train(path):
    d = torch.load(path, map_location="cpu", weights_only=False)
    classes = list(d["classes"])
    labels = d["labels"].long()
    original = d["original"]
    typographic = d["typographic"]
    if typographic.dim() == 2:
        typographic = typographic.unsqueeze(0)  # (1, N, 512): a single fixed attacked version (authors' setup)
    assert original.shape[0] == labels.shape[0] == typographic.shape[1]
    return classes, labels, original, typographic, d.get("meta", {})


def text_features(model, prompts, emb=None, star=None):
    toks = clip.tokenize(prompts).to(device)
    if emb is None:
        f = model.encode_text(toks)
    else:
        f = model.encode_text_with_learnt_tokens(toks, star, emb, is_emb=True)
    return F.normalize(f, dim=-1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train_file", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch_size", type=int, default=512)
    ap.add_argument("--lr", type=float, default=0.002)
    ap.add_argument("--eta_min", type=float, default=0.00005)
    ap.add_argument("--gamma", type=float, default=3.0)
    ap.add_argument("--seta", type=float, default=1.0)
    ap.add_argument("--dp_dtype", choices=["fp16", "fp32"], default="fp16")
    ap.add_argument("--no_defense", action="store_true", help="identity (KL) loss only")
    ap.add_argument("--no_reg", action="store_true", help="defense (CE) loss only")
    ap.add_argument("--extra_words", default=None, help="text file, one extra distractor word per line")
    ap.add_argument("--max_steps", type=int, default=0, help="stop early (smoke test)")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    set_seed(args.seed)

    model, _ = clip.load("ViT-B/32", device=device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    star = clip.tokenize(["*"]).to(device)[0][1]

    classes, labels, original, typographic, train_meta = load_train(args.train_file)
    extra = []
    if args.extra_words:
        with open(args.extra_words) as f:
            extra = [w.strip() for w in f if w.strip() and w.strip() not in classes]
    candidates = classes + extra  # extra words can only be wrong answers
    n, k_versions = labels.shape[0], typographic.shape[0]
    print(f"train images {n} | classes {len(classes)} | extra distractors {len(extra)} | attacked versions {k_versions}")

    # DP vector, exactly like the authors' main(): N(0, 0.02) with the model's dtype unless fp32 is requested
    dtype = model.dtype if args.dp_dtype == "fp16" else torch.float32
    init = torch.empty(1, 512, dtype=dtype, device=device)
    nn.init.normal_(init, std=0.02)
    emb = nn.Embedding.from_pretrained(init, freeze=False)
    opt = torch.optim.SGD(emb.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, args.epochs, eta_min=args.eta_min)

    logit_scale = model.logit_scale.exp().detach()
    gen = torch.Generator().manual_seed(args.seed)
    history, step = [], 0
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for ep in range(args.epochs):
        version = int(torch.randint(k_versions, (1,), generator=gen)) if k_versions > 1 else 0
        order = torch.randperm(n, generator=gen)
        sums = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "acc_attack_dp": 0.0}
        nb = 0
        for i in range(0, n, args.batch_size):
            idx = order[i:i + args.batch_size]
            x_clean = original[idx].to(device)
            x_typo = typographic[version, idx].to(device)
            y = labels[idx].to(device)

            template = random.choice(imagenet_templates)
            with torch.no_grad():
                t_orig = text_features(model, [template.format(c) for c in candidates])
            t_dp = text_features(model, [template.format("* " + c) for c in candidates], emb, star)

            logits_attack = logit_scale * x_typo @ t_dp.t()
            logits_orig = logit_scale * x_clean @ t_orig.t()
            logits_reg = logit_scale * x_clean @ t_dp.t()
            ce = F.cross_entropy(logits_attack.float(), y)
            kl = F.kl_div(F.log_softmax(logits_reg.float(), dim=-1), logits_orig.float().softmax(dim=-1),
                          reduction="batchmean")
            if args.no_defense:
                loss = kl
            elif args.no_reg:
                loss = ce
            else:
                loss = args.seta * ce + args.gamma * kl

            opt.zero_grad()
            loss.backward()
            opt.step()

            sums["loss"] += loss.item(); sums["ce"] += ce.item(); sums["kl"] += kl.item()
            sums["acc_attack_dp"] += (logits_attack.argmax(-1) == y).float().mean().item()
            nb += 1; step += 1
            if args.max_steps and step >= args.max_steps:
                break
        lr_used = sched.get_last_lr()[0]
        sched.step()
        rec = {"epoch": ep + 1, "lr": lr_used, "attack_version": version, "steps": nb,
               **{k: round(v / max(nb, 1), 5) for k, v in sums.items()}, "elapsed_s": round(time.time() - t0, 1)}
        history.append(rec)
        print(json.dumps(rec))
        if args.max_steps and step >= args.max_steps:
            break

    token = emb.weight.detach().to(model.dtype).cpu()  # shape (1, 512), same as the authors' saved token
    torch.save(token, os.path.join(args.out, "dp_token.pt"))
    meta = {
        "args": vars(args), "train_meta": train_meta, "n_train": n, "n_candidates": len(candidates),
        "train_seconds": round(time.time() - t0, 1),
        "peak_gpu_mem_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if device == "cuda" else None,
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
        "torch": torch.__version__, "history": history,
    }
    with open(os.path.join(args.out, "train_log.json"), "w") as f:
        json.dump(meta, f, indent=1)
    print("saved", os.path.join(args.out, "dp_token.pt"), "| train seconds", meta["train_seconds"],
          "| peak MB", meta["peak_gpu_mem_mb"])


if __name__ == "__main__":
    main()

Writing train_dp_cached.py


In [4]:
%%writefile eval_cached.py
"""Evaluate CLIP and CLIP + Defense-Prefix on cached image features.

Text features and scoring follow the authors' eval.py exactly:
one template per dataset (templates[0]), "* {class}" with the * embedding
replaced by the DP vector, cosine similarity x 100, argmax.

Usage (from the Defense-Prefix repo root):
  python eval_cached.py --cache CACHE_DIR --token learned_token/dp_vit-b32.pt --tag authors_token
"""
import argparse
import csv
import glob
import json
import os
import sys

import clip
import torch
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
from utils.non_nv import encode_text_with_learnt_tokens  # authors' function

device = "cuda" if torch.cuda.is_available() else "cpu"

ORDER = ["imagenet", "caltech", "pets", "cars", "flowers", "food", "aircraft", "dtd", "sun", "eurosat",
         "disentangling", "paint", "rta-100"]
# Paper, Tables B, C and 3 (ViT-B/32): (CLIP, CLIP + DP)
PAPER = {
    "clean": {"imagenet": (62.02, 62.48), "caltech": (88.64, 89.28), "pets": (87.35, 87.22), "cars": (58.72, 57.47),
              "flowers": (66.32, 63.82), "food": (84.14, 83.65), "aircraft": (18.99, 19.26), "dtd": (44.57, 40.64),
              "sun": (61.74, 61.41), "eurosat": (42.98, 43.85)},
    "synthetic": {"imagenet": (39.10, 49.83), "caltech": (63.97, 79.54), "pets": (58.95, 72.88), "cars": (21.02, 28.64),
                  "flowers": (31.32, 44.12), "food": (56.27, 67.79), "aircraft": (10.83, 14.49), "dtd": (25.53, 31.6),
                  "sun": (34.02, 43.50), "eurosat": (4.86, 9.65)},
    "real": {"disentangling": (43.27, 71.93), "paint": (50.00, 63.64), "rta-100": (47.20, 58.00)},
}


def load_model():
    model, _ = clip.load("ViT-B/32", device=device)
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    model.eval()
    return model


@torch.no_grad()
def class_text_features(model, classes, template, token=None):
    if token is None:
        toks = torch.cat([clip.tokenize(template.format(c)) for c in classes]).to(device)
        f = model.encode_text(toks)
        return f / f.norm(dim=-1, keepdim=True)
    toks = torch.cat([clip.tokenize(template.format(f"* {c}")) for c in classes]).to(device)
    star = clip.tokenize(["*"]).to(device)[0][1]
    f = model.encode_text_with_learnt_tokens(toks, star, token.unsqueeze(0), is_emb=False)
    return F.normalize(f, dim=-1)


@torch.no_grad()
def predictions(img_feats, text_feats, bs=8192):
    out = []
    for i in range(0, len(img_feats), bs):
        sim = 100.0 * (img_feats[i:i + bs].to(device) @ text_feats.T)
        out.append(sim.argmax(-1).cpu())
    return torch.cat(out)


def evaluate(model, cache_dir, token):
    """Returns rows: dataset, condition, n, acc_clip, acc_dp, plus per-image correctness for later stats."""
    rows, per_image = [], {}
    files = {os.path.basename(p)[:-3]: p for p in glob.glob(os.path.join(cache_dir, "*.pt"))}
    for name in [d for d in ORDER if d in files]:
        c = torch.load(files[name], map_location="cpu", weights_only=False)
        t_clip = class_text_features(model, c["classes"], c["template"])
        t_dp = class_text_features(model, c["classes"], c["template"], token)
        conds = [("real", c["clean"])] if c["real_world"] else [("clean", c["clean"]), ("synthetic", c["attack"])]
        for cond, feats in conds:
            y = c["labels"]
            ok_clip = predictions(feats, t_clip).eq(y)
            ok_dp = predictions(feats, t_dp).eq(y)
            per_image[f"{name}/{cond}"] = {"clip": ok_clip, "dp": ok_dp}
            rows.append({"dataset": name, "condition": cond, "n": len(y),
                         "acc_clip": round(100 * ok_clip.float().mean().item(), 2),
                         "acc_dp": round(100 * ok_dp.float().mean().item(), 2)})
    return rows, per_image


def with_paper(rows):
    for r in rows:
        p = PAPER.get(r["condition"], {}).get(r["dataset"])
        r["paper_clip"], r["paper_dp"] = (p if p else (None, None))
    return rows


def print_table(rows):
    print(f"{'dataset':<14}{'cond':<10}{'n':>7} | {'paper CLIP':>10}{'ours CLIP':>10} | {'paper DP':>9}{'ours DP':>9}")
    for r in rows:
        pc = "" if r["paper_clip"] is None else f"{r['paper_clip']:.2f}"
        pd = "" if r["paper_dp"] is None else f"{r['paper_dp']:.2f}"
        print(f"{r['dataset']:<14}{r['condition']:<10}{r['n']:>7} | {pc:>10}{r['acc_clip']:>10.2f} | {pd:>9}{r['acc_dp']:>9.2f}")
    for cond in ["clean", "synthetic", "real"]:
        sub = [r for r in rows if r["condition"] == cond]
        if sub:
            mc = sum(r["acc_clip"] for r in sub) / len(sub)
            md = sum(r["acc_dp"] for r in sub) / len(sub)
            print(f"  mean {cond:<9} over {len(sub)} sets: CLIP {mc:.2f} | DP {md:.2f}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache", required=True)
    ap.add_argument("--token", default="learned_token/dp_vit-b32.pt")
    ap.add_argument("--tag", default="authors_token")
    ap.add_argument("--out", default=None, help="results folder (default: <cache>/../results)")
    args = ap.parse_args()

    token = torch.load(args.token, map_location=device, weights_only=False)
    token = token.detach()
    if token.dim() == 1:          # authors save shape (1, 512); eval.py passes token.unsqueeze(0)
        token = token.unsqueeze(0)
    model = load_model()
    token = token.to(device).type(model.dtype)

    rows, per_image = evaluate(model, args.cache, token)
    rows = with_paper(rows)
    print_table(rows)

    out = args.out or os.path.join(os.path.dirname(os.path.abspath(args.cache)), "results")
    os.makedirs(out, exist_ok=True)
    with open(os.path.join(out, f"{args.tag}.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    torch.save(per_image, os.path.join(out, f"{args.tag}_per_image.pt"))
    print("saved", os.path.join(out, f"{args.tag}.csv"))


if __name__ == "__main__":
    main()

Writing eval_cached.py


In [5]:
%%writefile eval_seen_unseen.py
"""Evaluate CLIP and one or more DP tokens on the seen/unseen test sets from make_seen_unseen.py.

Scoring is the authors' eval.py rule (dataset template, "* {class}" with * -> DP vector,
cosine x 100, argmax) over the shared candidate list (dataset classes + seen + unseen words).

Reported per dataset and token:
  acc_clean, acc_seen, acc_unseen         top-1 accuracy (%)
  hit_seen, hit_unseen                    % of images predicted as the written word (attack success)
  gap = acc_seen - acc_unseen             paired over the same images, 95% bootstrap CI
  For DP tokens additionally the difference-in-differences
  did = (DP - CLIP)_seen - (DP - CLIP)_unseen, which removes any difference in raw attack strength.

Usage (from the Defense-Prefix repo root):
  python eval_seen_unseen.py --data cache_su/su_pets.pt cache_su/su_caltech.pt \
      --tokens authors=learned_token/dp_vit-b32.pt seed0=runs/dp_seed0/dp_token.pt --out results/seen_unseen
"""
import argparse
import csv
import json
import os
import sys

import torch
import torch.nn.functional as F

sys.path.insert(0, os.getcwd())
import clip  # noqa: E402
from utils.non_nv import encode_text_with_learnt_tokens  # noqa: E402

device = "cuda" if torch.cuda.is_available() else "cpu"


@torch.no_grad()
def text_feats(model, names, template, token=None):
    if token is None:
        f = model.encode_text(torch.cat([clip.tokenize(template.format(c)) for c in names]).to(device))
        return F.normalize(f, dim=-1)
    toks = torch.cat([clip.tokenize(template.format(f"* {c}")) for c in names]).to(device)
    star = clip.tokenize(["*"]).to(device)[0][1]
    return F.normalize(model.encode_text_with_learnt_tokens(toks, star, token.unsqueeze(0), is_emb=False), dim=-1)


@torch.no_grad()
def predict(x, t):
    return torch.cat([(100.0 * x[i:i + 8192].to(device) @ t.T).argmax(-1).cpu() for i in range(0, len(x), 8192)])


def boot_ci(diff, n_boot=2000, seed=0):
    """95% percentile bootstrap CI of the mean of a per-image difference vector (resampling images)."""
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(len(diff), (n_boot, len(diff)), generator=g)
    means = diff[idx].mean(1)
    return round(100 * means.quantile(0.025).item(), 2), round(100 * means.quantile(0.975).item(), 2)


def load_token(path, dtype):
    t = torch.load(path, map_location=device, weights_only=False).detach()
    if t.dim() == 1:
        t = t.unsqueeze(0)
    return t.to(device).type(dtype)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", nargs="+", required=True)
    ap.add_argument("--tokens", nargs="+", required=True, help="name=path pairs")
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    model, _ = clip.load("ViT-B/32", device=device)
    model.eval()
    model.encode_text_with_learnt_tokens = type(model.encode_text)(encode_text_with_learnt_tokens, model)
    tokens = {k: load_token(v, model.dtype) for k, v in (s.split("=", 1) for s in args.tokens)}

    rows = []
    for path in args.data:
        d = torch.load(path, map_location="cpu", weights_only=False)
        cand, y = d["candidates"], d["labels"]
        w_seen = torch.tensor([cand.index(w) for w in d["word_seen"]])
        w_unseen = torch.tensor([cand.index(w) for w in d["word_unseen"]])
        correct = {}
        for name, tok in [("clip", None)] + list(tokens.items()):
            t = text_feats(model, cand, d["template"], tok)
            p = {k: predict(d[k], t) for k in ("clean", "seen", "unseen")}
            ok = {k: p[k].eq(y).float() for k in p}
            correct[name] = ok
            gap = ok["seen"] - ok["unseen"]
            r = {"dataset": d["dataset"], "model": name, "n": len(y),
                 "acc_clean": round(100 * ok["clean"].mean().item(), 2),
                 "acc_seen": round(100 * ok["seen"].mean().item(), 2),
                 "acc_unseen": round(100 * ok["unseen"].mean().item(), 2),
                 "hit_seen": round(100 * p["seen"].eq(w_seen).float().mean().item(), 2),
                 "hit_unseen": round(100 * p["unseen"].eq(w_unseen).float().mean().item(), 2),
                 "gap_seen_minus_unseen": round(100 * gap.mean().item(), 2)}
            r["gap_ci95"] = boot_ci(gap)
            if name != "clip":
                did = (ok["seen"] - correct["clip"]["seen"]) - (ok["unseen"] - correct["clip"]["unseen"])
                r["did"] = round(100 * did.mean().item(), 2)
                r["did_ci95"] = boot_ci(did)
            rows.append(r)
            print(json.dumps(r))

    keys = list(dict.fromkeys(k for r in rows for k in r))
    with open(os.path.join(args.out, "seen_unseen.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print("saved", os.path.join(args.out, "seen_unseen.csv"))


if __name__ == "__main__":
    main()

Writing eval_seen_unseen.py


In [6]:
%%writefile summarize_runs.py
"""Aggregate eval_cached.py CSVs over seeds: mean and sample std per dataset/condition, next to the paper.

Usage:
  python summarize_runs.py --results RESULTS_DIR --group repro=dp_seed0,dp_seed1,dp_seed2 --group authors=authors_token
Each name in a group is a tag written by eval_cached.py (RESULTS_DIR/<tag>.csv).
"""
import argparse
import csv
import os
import statistics as st


def read(path):
    with open(path) as f:
        return list(csv.DictReader(f))


def fmt(vals):
    m = st.mean(vals)
    return f"{m:.2f} ± {st.stdev(vals):.2f}" if len(vals) > 1 else f"{m:.2f}"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", required=True)
    ap.add_argument("--group", action="append", required=True, help="name=tag1,tag2,...")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    groups = {}
    for g in args.group:
        name, tags = g.split("=", 1)
        groups[name] = [read(os.path.join(args.results, f"{t}.csv")) for t in tags.split(",")]

    first = next(iter(groups.values()))[0]
    keys = [(r["dataset"], r["condition"]) for r in first]
    header = ["dataset", "condition", "paper_clip", "paper_dp", "ours_clip"] + [f"{g}_dp" for g in groups]
    lines = []
    for ds, cond in keys:
        row = {"dataset": ds, "condition": cond}
        ref = next(r for r in first if r["dataset"] == ds and r["condition"] == cond)
        row["paper_clip"], row["paper_dp"], row["ours_clip"] = ref["paper_clip"], ref["paper_dp"], ref["acc_clip"]
        for g, runs in groups.items():
            vals = [float(next(r for r in run if r["dataset"] == ds and r["condition"] == cond)["acc_dp"]) for run in runs]
            row[f"{g}_dp"] = fmt(vals)
        lines.append(row)
    # condition means: mean over datasets within each run, then mean ± std over runs
    for cond in ["clean", "synthetic", "real"]:
        sub = [k for k in keys if k[1] == cond]
        if not sub:
            continue
        row = {"dataset": f"MEAN ({len(sub)} sets)", "condition": cond, "paper_clip": "", "paper_dp": ""}
        row["ours_clip"] = f"{st.mean(float(next(r for r in first if (r['dataset'], r['condition']) == k)['acc_clip']) for k in sub):.2f}"
        for g, runs in groups.items():
            per_run = [st.mean(float(next(r for r in run if (r["dataset"], r["condition"]) == k)["acc_dp"]) for k in sub)
                       for run in runs]
            row[f"{g}_dp"] = fmt(per_run)
        lines.append(row)

    width = [max(len(h), *(len(str(l.get(h, ""))) for l in lines)) for h in header]
    print("  ".join(h.ljust(w) for h, w in zip(header, width)))
    for l in lines:
        print("  ".join(str(l.get(h, "")).ljust(w) for h, w in zip(header, width)))
    if args.out:
        with open(args.out, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=header)
            w.writeheader()
            w.writerows(lines)
        print("saved", args.out)


if __name__ == "__main__":
    main()

Writing summarize_runs.py


In [7]:
# A4) Smoke test: 5 training steps, then make sure the saved token loads in eval_cached.py
!python train_dp_cached.py --train_file {TRAIN} --out {W}/runs/smoke --seed 0 --max_steps 5
!python eval_cached.py --cache {CACHE} --token {W}/runs/smoke/dp_token.pt --tag smoke --out {W}/results | tail -4

100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 105MiB/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
train images 130000 | classes 100 | extra distractors 0 | attacked versions 1
{"epoch": 1, "lr": 0.002, "attack_version": 0, "steps": 5, "loss": 2.14517, "ce": 1.86741, "kl": 0.09259, "acc_attack_dp": 0.50234, "elapsed_s": 2.7}
saved /content/drive/MyDrive/dp_project/experiments/runs/smoke/dp_token.pt | train seconds 2.7 | peak MB 1916.3
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
  mean clean     over 7 sets: CLIP 61.92 | DP 60.22
  mean synthetic over 7 sets: CLIP 35.92 | DP 36.78
  mean real      over 3 sets: CLIP 46.26 | DP 54.05
saved /content/drive/MyDrive/dp_project/experiments/results/smoke.csv


## C) Step 4 — re-train the DP vector (authors' settings) with 3 seeds, plus two controls
- `dp_seed{0,1,2}`: full method (CE + 3·KL), fp16 DP parameter as in the authors' code.
- `ctrl_identity_only`: identity (KL) loss only — the paper's "w/o defense loss" row (Table 9).
- `ctrl_random`: an untrained N(0, 0.02) vector — does *any* extra token change accuracy?

In [8]:
# C1) Three seeds of the full method (a finished seed is skipped, so the cell can be re-run after a disconnect)
for seed in [0, 1, 2]:
    if os.path.exists(f"{W}/runs/dp_seed{seed}/dp_token.pt"):
        print("seed", seed, "already trained"); continue
    !python train_dp_cached.py --train_file {TRAIN} --out {W}/runs/dp_seed{seed} --seed {seed} 2>&1 | tail -3

{"epoch": 9, "lr": 0.00023620843048442636, "attack_version": 0, "steps": 254, "loss": 1.51447, "ce": 1.38224, "kl": 0.04408, "acc_attack_dp": 0.59823, "elapsed_s": 680.3}
{"epoch": 10, "lr": 9.771989661222532e-05, "attack_version": 0, "steps": 254, "loss": 1.49954, "ce": 1.36921, "kl": 0.04344, "acc_attack_dp": 0.60188, "elapsed_s": 756.0}
saved /content/drive/MyDrive/dp_project/experiments/runs/dp_seed0/dp_token.pt | train seconds 756.0 | peak MB 1916.3
{"epoch": 9, "lr": 0.00023620843048442636, "attack_version": 0, "steps": 254, "loss": 1.51517, "ce": 1.37012, "kl": 0.04835, "acc_attack_dp": 0.60238, "elapsed_s": 684.2}
{"epoch": 10, "lr": 9.771989661222532e-05, "attack_version": 0, "steps": 254, "loss": 1.51625, "ce": 1.36616, "kl": 0.05003, "acc_attack_dp": 0.60354, "elapsed_s": 760.0}
saved /content/drive/MyDrive/dp_project/experiments/runs/dp_seed1/dp_token.pt | train seconds 760.0 | peak MB 1916.3
{"epoch": 9, "lr": 0.00023620843048442636, "attack_version": 0, "steps": 254, "los

In [9]:
# C2) Controls
if not os.path.exists(f"{W}/runs/ctrl_identity_only/dp_token.pt"):
    !python train_dp_cached.py --train_file {TRAIN} --out {W}/runs/ctrl_identity_only --seed 0 --no_defense 2>&1 | tail -2
torch.manual_seed(0)
rand = torch.empty(1, 512); torch.nn.init.normal_(rand, std=0.02)
os.makedirs(f"{W}/runs/ctrl_random", exist_ok=True); torch.save(rand.half(), f"{W}/runs/ctrl_random/dp_token.pt")
for r in sorted(glob.glob(f"{W}/runs/*/train_log.json")):
    m = json.load(open(r)); print(r.split("/")[-2], "| seconds", m["train_seconds"], "| peak MB", m["peak_gpu_mem_mb"], "| last epoch", m["history"][-1])

{"epoch": 10, "lr": 9.771989661222532e-05, "attack_version": 0, "steps": 254, "loss": 0.01227, "ce": 2.08195, "kl": 0.01227, "acc_attack_dp": 0.45868, "elapsed_s": 759.8}
saved /content/drive/MyDrive/dp_project/experiments/runs/ctrl_identity_only/dp_token.pt | train seconds 759.8 | peak MB 1917.0
ctrl_identity_only | seconds 759.8 | peak MB 1917.0 | last epoch {'epoch': 10, 'lr': 9.771989661222532e-05, 'attack_version': 0, 'steps': 254, 'loss': 0.01227, 'ce': 2.08195, 'kl': 0.01227, 'acc_attack_dp': 0.45868, 'elapsed_s': 759.8}
dp_seed0 | seconds 756.0 | peak MB 1916.3 | last epoch {'epoch': 10, 'lr': 9.771989661222532e-05, 'attack_version': 0, 'steps': 254, 'loss': 1.49954, 'ce': 1.36921, 'kl': 0.04344, 'acc_attack_dp': 0.60188, 'elapsed_s': 756.0}
dp_seed1 | seconds 760.0 | peak MB 1916.3 | last epoch {'epoch': 10, 'lr': 9.771989661222532e-05, 'attack_version': 0, 'steps': 254, 'loss': 1.51625, 'ce': 1.36616, 'kl': 0.05003, 'acc_attack_dp': 0.60354, 'elapsed_s': 760.0}
dp_seed2 | sec

## D) Evaluation of every token

In [10]:
# D1) All cached test sets (same scoring as the authors' eval.py)
!python eval_cached.py --cache {CACHE} --token learned_token/dp_vit-b32.pt --tag authors_token --out {W}/results > {W}/results/eval_authors_token.txt
for tag in ["dp_seed0", "dp_seed1", "dp_seed2", "ctrl_identity_only", "ctrl_random"]:
    !python eval_cached.py --cache {CACHE} --token {W}/runs/{tag}/dp_token.pt --tag {tag} --out {W}/results > {W}/results/eval_{tag}.txt
!python summarize_runs.py --results {W}/results --group authors=authors_token --group repro=dp_seed0,dp_seed1,dp_seed2 --group identity_only=ctrl_identity_only --group random=ctrl_random --out {W}/results/summary_step4.csv

/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export

In [11]:
# D2) Seen / unseen with all tokens (test sets made in notebook 04)
SU = " ".join(sorted(glob.glob(f"{W}/cache_su/su_*.pt")))
assert SU, "run notebook 04 first"
toks = "authors=learned_token/dp_vit-b32.pt " + " ".join(f"{t}={W}/runs/{t}/dp_token.pt" for t in ["dp_seed0", "dp_seed1", "dp_seed2", "ctrl_identity_only", "ctrl_random"])
!python eval_seen_unseen.py --data {SU} --tokens {toks} --out {W}/results/seen_unseen_all

/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
{"dataset": "caltech", "model": "clip", "n": 8677, "acc_clean": 81.15, "acc_seen": 58.57, "acc_unseen": 53.76, "hit_seen": 19.57, "hit_unseen": 29.13, "gap_seen_minus_unseen": 4.81, "gap_ci95": [3.77, 5.84]}
{"dataset": "caltech", "model": "authors", "n": 8677, "acc_clean": 82.98, "acc_seen": 72.87, "acc_unseen": 69.85, "hit_seen": 4.56, "hit_unseen": 10.31, "gap_seen_minus_unseen": 3.02, "gap_ci95": [2.19, 3.9], "did": -1.79, "did_ci95": [-2.86, -0.68]}
{"dataset": "caltech", "model": "dp_seed0", "n": 8677, "acc_clean": 82.1, "acc_seen": 73.49, "acc_unseen": 70.39, "hit_seen": 3.8, "hit_unseen": 8.99, "gap_seen_minus_unseen": 3.1, "gap_ci95": [2.29, 3.91], "did": -1.71, "did_ci95": [-2.84, -0.59]}
{"dataset": "caltech", "model": "dp_seed1", "n": 8677, "acc_clean": 82.27, "acc_seen": 72.99, "acc_unseen": 70.44, "hit_se

In [12]:
# D3) Pack the small outputs (results, logs, tokens, example images) into one zip in Drive
!cd {W} && zip -q -r results_05_06.zip results runs cache_su/*.json cache_su/examples_* && ls -la {W}

total 3716
drwx------ 5 root root    4096 Nov 12 18:15 .
drwx------ 8 root root    4096 Nov  5 17:14 ..
drwx------ 4 root root    4096 Nov 12 16:21 cache_su
-rw------- 1 root root   14260 Nov 12 17:11 pip_freeze_step3_4.txt
drwx------ 5 root root    4096 Nov 12 18:15 results
-rw------- 1 root root 3769730 Nov 12 18:15 results_05_06.zip
drwx------ 8 root root    4096 Nov 12 18:04 runs
